# Multi-GPU Distributed Data Parallel (DDP) Training on Kaggle 2x Tesla T4
## 114M Parameter Banach Contraction LLaMA on WikiText-103

This notebook trains and benchmarks 100M+ parameter models with large effective batch size (`65,536 tokens/step`) across 2x Tesla T4 GPUs:
1. **Baseline**: Standard 3-matrix SwiGLU LLaMA-114M
2. **Champion**: Plain 2-Matrix MLP + Spectral Barrier Regularization LLaMA-114M

Both models are trained dynamically until validation loss plateau.

In [ ]:
# 1. Verify 2x Tesla T4 GPU Environment
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Install / Upgrade Dependencies
!pip install -q tiktoken datasets transformers

In [ ]:
# 3. Clone Repository (if running standalone in Kaggle) or use local working directory
import os
if not os.path.exists("llama"):
    !git clone https://github.com/phpproger/great.git
    %cd great
!ls -la

In [ ]:
# 4. Launch Multi-GPU DDP Training with torchrun (2x Tesla T4, utilizes ~11.6 GB VRAM each)
# Effective batch size = 2 GPUs * 12 batch * 4 accum = 96 sequences = 49,152 tokens/step
!torchrun --nproc_per_node=2 train_ddp.py \
    --arch compare \
    --model_size 114m \
    --seq_len 512 \
    --batch_size 12 \
    --grad_accum 4 \
    --lr 4e-4 \
    --min_steps 400 \
    --max_steps 4000 \
    --eval_interval 50 \
    --patience 8 \
    --min_delta 0.001 \
    --output_dir checkpoints_ddp

In [ ]:
# 5. Inspect Checkpoints and Outputs
!ls -lh checkpoints_ddp/

In [ ]:
# 6. Build Comparison Curves and Executive Performance Matrix
# Automatically loads history from checkpoints_ddp/ and plots:
# 1. Validation Perplexity (PPL) vs Step (with GPT-2 Small reference)
# 2. Validation Cross-Entropy Loss vs Step
# 3. Smoothed Training Loss Dynamics & Stability
# 4. Learning Rate Schedule & Convergence Points

import os
from plot_comparison import load_architecture_history, plot_comparison_curves, print_executive_summary, LABEL_MAP

output_dir = "checkpoints_ddp"
known_archs = ["swiglu_baseline", "plain_spectral", "swiglu_aux", "swiglu_strong_aux", "swiglu_cycle", "swiglu_k2_aux", "plain_spectral_cycle", "plain_strong_cycle"]
discovered = list(known_archs)
if os.path.exists(output_dir):
    for f in os.listdir(output_dir):
        if f.endswith("_history.json"):
            name = f[:-len("_history.json")]
            if name not in discovered:
                discovered.append(name)
        elif f.endswith("_114m_best.pt"):
            name = f[:-len("_114m_best.pt")]
            if name not in discovered:
                discovered.append(name)

histories = {}
for a in discovered:
    h = load_architecture_history(output_dir, a)
    if h and len(h.get("val_steps", [])) > 0:
        label = LABEL_MAP.get(a, a)
        histories[label] = h

# Display formatted comparison matrix
print_executive_summary(histories)

# Render publication-grade comparison figures inline
if histories:
    plot_comparison_curves(
        histories,
        save_path=os.path.join(output_dir, "comparison_curves.png"),
        show_plot=True
    )

In [ ]:
# 7. Multi-Layer Banach Contraction Inversion Benchmark (Anderson vs Picard)
# Directly evaluates multi-layer invertibility (h_12 -> h_0) on both trained 114M models:
# Computes relative reconstruction error ||h_rec - h_0||_2 / ||h_0||_2 across all 12 layers.

import os
import math
import torch
from llama.config import LlamaConfig
from llama.model import LlamaForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_swi_path = "checkpoints_ddp/swiglu_baseline_114m_best.pt"
ckpt_plain_path = "checkpoints_ddp/plain_spectral_114m_best.pt"

models_to_test = {}
if os.path.exists(ckpt_swi_path):
    cfg_swi = LlamaConfig.llama_114m(vocab_size=50257)
    cfg_swi.mlp_type = "swiglu"
    cfg_swi.use_spectral_barrier = False
    m_swi = LlamaForCausalLM(cfg_swi).to(device)
    ckpt_s = torch.load(ckpt_swi_path, map_location=device)
    m_swi.load_state_dict(ckpt_s["model_state_dict"] if "model_state_dict" in ckpt_s else ckpt_s)
    m_swi.eval()
    models_to_test["0. SwiGLU Baseline"] = m_swi

if os.path.exists(ckpt_plain_path):
    cfg_pl = LlamaConfig.llama_114m(vocab_size=50257)
    cfg_pl.mlp_type = "plain"
    cfg_pl.use_spectral_barrier = True
    m_pl = LlamaForCausalLM(cfg_pl).to(device)
    ckpt_p = torch.load(ckpt_plain_path, map_location=device)
    m_pl.load_state_dict(ckpt_p["model_state_dict"] if "model_state_dict" in ckpt_p else ckpt_p)
    m_pl.eval()
    models_to_test["1. Plain MLP + Spectral"] = m_pl

if models_to_test:
    print("=" * 85)
    print("  MULTI-LAYER BANACH INVERSION BENCHMARK (12 LAYERS, 768 HIDDEN, 114M PARAMS)")
    print("=" * 85)
    torch.manual_seed(42)
    sample_tokens = torch.randint(0, 50257, (4, 128), device=device)
    print(f"  {'Model Architecture':<32} | {'Picard k=1':<12} | {'Picard k=2':<12} | {'Anderson Accel':<15}")
    print("-" * 85)

    for name, model in models_to_test.items():
        with torch.no_grad():
            h_in = model.model.embed_tokens(sample_tokens)
            if getattr(model.config, "scale_embeddings", False):
                h_in = h_in * math.sqrt(model.config.hidden_size)
            seq_l = sample_tokens.shape[1]
            cos, sin = model.model.rotary_emb(h_in, seq_l)
            h = h_in
            for layer in model.model.layers:
                h, _ = layer(h, cos=cos, sin=sin)
            
            # Picard k=1
            h_p1 = h.clone()
            for layer in reversed(model.model.layers):
                h_p1 = layer.invert(h_p1, cos=cos, sin=sin, k_steps=1, method="picard")
            err_p1 = (h_p1 - h_in).norm().item() / max(h_in.norm().item(), 1e-6)

            # Picard k=2
            h_p2 = h.clone()
            for layer in reversed(model.model.layers):
                h_p2 = layer.invert(h_p2, cos=cos, sin=sin, k_steps=2, method="picard")
            err_p2 = (h_p2 - h_in).norm().item() / max(h_in.norm().item(), 1e-6)

            # Anderson Acceleration (m=2)
            h_and = h.clone()
            for layer in reversed(model.model.layers):
                h_and = layer.invert(h_and, cos=cos, sin=sin, method="anderson")
            err_and = (h_and - h_in).norm().item() / max(h_in.norm().item(), 1e-6)

            print(f"  {name:<32} | {err_p1:<12.4f} | {err_p2:<12.4f} | {err_and:<15.4f}")
    print("=" * 85)
else:
    print("[*] No checkpoints found yet in checkpoints_ddp/. Run training in Step 4 first.")
